# نموذج مطابقة قواعد حوكمة الجمعيات والمؤسسات الأهلية باللائحة التنفيذية

**المدخل الوحيد:** `legal_articles_full.csv` — مواد الوثيقتين بعد التنظيف (47 مادة من قواعد الحوكمة، 129 من اللائحة التنفيذية).

**ما يفعله:**
1. **المطابقة بنموذج المعنى BAAI/bge-m3 (الطريقة المعتمدة):** لكل مادة حوكمة أقرب 3 مواد من اللائحة، مع درجة التشابه وهل هي فوق الحد المعتمد 0.75. يقرأ عمود `text_raw` والمادة كاملة دون قص.
2. **خط الأساس TF-IDF** (حروف 2–4، على عمود `text`) بالطريقة نفسها للمقارنة.
3. **التجميع الموضوعي:** KMeans على تمثيلات bge-m3، وعدد المجموعات بأعلى silhouette، ومقارنتها بتصنيف المواضيع (ARI).
4. **أداة العرض (آخر خلية):** اكتبي نص مادة، فيقترح أقرب مواد اللائحة.

التوائم: 26 زوجًا من مواد اللائحة نصها متطابق تقريبًا (نسخة للجمعية ونسخة للمؤسسة، أو للمجلس وللفرع) تُعرض مادة واحدة مثل «79/107».

**نتائج تقييم النموذج بالمراجعة البشرية** (4 مراجِعات، 141 مقترحًا لـ bge-m3؛ التفاصيل في `outputs/نتائج_النموذج.xlsx`، ولا يُعاد حسابها هنا):
- الدقة المرنة (بين المقترحات الثلاثة مادة مطابقة أو مطابقة جزئيًا): bge-m3 41/47 = 87% [75%–94%]، TF-IDF 40/47 = 85% [72%–93%]. الصارمة (مطابقة تامة): 23/47 = 49% للطريقتين. الفرق بين الطريقتين غير مثبت.
- حد التشابه 0.75 (أعلى Youden's J): 91% من المقترحات فوقه صحيحة؛ 21 مادة حوكمة لها مقترح فوقه. الحد مستخرج من العينة نفسها.
- الاتفاق بين المراجِعات ضعيف (Cohen's kappa = 0.13)؛ 12 حكمًا مختلفًا عليه حُسم بمراجعة الذكاء الاصطناعي (موثق في سجل AI).
- التجميع: k = 29، silhouette = 0.147 (منخفض)، ARI مع المواضيع = 0.28 (اتفاق جزئي).

**التشغيل في Google Colab:** Runtime ← Change runtime type ← GPU، ثم Runtime ← Run all، وارفعي `legal_articles_full.csv` عند الطلب.

كتب الأكواد Claude (نموذج ذكاء اصطناعي من Anthropic) بطلب من فجر؛ فجر حددت المطلوب واعتمدت القرارات.

In [ ]:
!pip -q install sentence-transformers scikit-learn openpyxl

## المدخل

In [ ]:
import re, html, zipfile, os, numpy as np, pandas as pd
from google.colab import files
up = files.upload()                                   # legal_articles_full.csv
CSV = [f for f in up if f.startswith('legal_articles')]
assert len(CSV) == 1, 'ارفعي legal_articles_full.csv'; CSV = CSV[0]
d = pd.read_csv(CSV, encoding='utf-8-sig')
assert len(d) == 176 and (d.doc_key == 'gov').sum() == 47 and (d.doc_key == 'reg').sum() == 129
TWINS = [(8,47),(10,49),(79,107),(13,52),(6,45),(39,69),(37,67),(31,61),(11,50),(14,53),(84,98),(77,94),(12,51),
         (87,112),(15,54),(40,70),(4,44),(32,63),(33,64),(74,91),(81,96),(38,68),(9,48),(76,93),(7,46),(86,111)]
THR = 0.75
print('المواد:', len(d))

## 1) خط الأساس: TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def main(csv, out):
    d = pd.read_csv(csv, encoding="utf-8-sig")
    gov = d[d.doc_key == "gov"].reset_index(drop=True)
    reg = d[d.doc_key == "reg"].reset_index(drop=True)
    assert len(gov) == 47 and len(reg) == 129
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True)
    mat = vec.fit_transform(pd.concat([gov["text"], reg["text"]]).tolist())
    S = (mat[:len(gov)] @ mat[len(gov):].T).toarray()
    reg_no = reg["article_no"].tolist()
    rep = {}
    for a, b in TWINS:
        rep[a] = rep[b] = f"{a}/{b}"
    rows = []
    for i, g in gov.iterrows():
        seen, picked, scores = set(), [], []
        for j in np.argsort(-S[i], kind="stable"):
            k = rep.get(reg_no[j], str(reg_no[j]))
            if k in seen:
                continue
            seen.add(k); picked.append(k); scores.append(float(S[i, j]))
            if len(picked) == 4:
                break
        for r in range(3):
            rows.append({"gov_article": g.article_no, "rank": r + 1, "reg_group": picked[r], "score": round(scores[r], 4)})
    pd.DataFrame(rows).to_csv(out, index=False, encoding="utf-8-sig")
    return S, gov, reg

main(CSV, 'tfidf_matches.csv')
print(pd.read_csv('tfidf_matches.csv').head(6).to_string(index=False))

## 2) الطريقة المعتمدة: bge-m3

In [ ]:
from sentence_transformers import SentenceTransformer
m = SentenceTransformer('BAAI/bge-m3')
lens = [len(m.tokenizer(t)['input_ids']) for t in d['text_raw']]
print('max_seq_length:', m.max_seq_length, '| أطول مادة:', max(lens), 'قطعة'); assert max(lens) <= m.max_seq_length
enc = lambda xs: m.encode(xs, batch_size=4, normalize_embeddings=True, show_progress_bar=False)
E = enc(d['text_raw'].tolist())                       # تمثيل المواد الـ176 (يُستعمل في المطابقة والتجميع والعرض)
np.save('bge_m3_embeddings.npy', E)
gi = d.index[d.doc_key == 'gov']; ri = d.index[d.doc_key == 'reg']
gov = d.loc[gi].reset_index(drop=True); reg = d.loc[ri].reset_index(drop=True)
S = E[gi] @ E[ri].T; R = E[ri]
reg_no = reg['article_no'].tolist(); rep = {}
for a, b in TWINS: rep[a] = rep[b] = f'{a}/{b}'
rows = []
for i, g in gov.iterrows():
    seen, picked, scores = set(), [], []
    for j in np.argsort(-S[i]):
        k = rep.get(reg_no[j], str(reg_no[j]))
        if k in seen: continue
        seen.add(k); picked.append(k); scores.append(float(S[i, j]))
        if len(picked) == 4: break
    for r in range(3):
        rows.append({'gov_article': g.article_no, 'rank': r + 1, 'reg_group': picked[r], 'score': round(scores[r], 4),
                     f'above_{THR}': scores[r] >= THR})
bm = pd.DataFrame(rows); bm.to_csv('bge_m3_matches.csv', index=False, encoding='utf-8-sig')
up_ = bm[bm[f'above_{THR}']]
print('مقترحات فوق الحد:', len(up_), '| مواد حوكمة لها مقترح فوق الحد:', up_.gov_article.nunique(), 'من 47')
print(bm.head(6).to_string(index=False))

## 3) التجميع الموضوعي (KMeans + silhouette، ومقارنته بالمواضيع بـ ARI)

In [ ]:
import sys
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
SEED, K_RANGE = 20261028, range(2, 31)
DOC = {"gov": "الحوكمة", "reg": "اللائحة"}


def main(csv="legal_articles_full_cleaned_v2.csv", npy="bge_m3_embeddings.npy", from_npy=False, embed_fn=None, out_prefix=""):
    d = pd.read_csv(csv, encoding="utf-8-sig")
    assert len(d) == 176 and (d.doc_key == "gov").sum() == 47 and (d.doc_key == "reg").sum() == 129
    if from_npy:
        E = np.load(npy); print("المتجهات من", npy)
    else:
        E = np.asarray(embed_fn(d["text_raw"].tolist()), dtype=float); np.save(out_prefix + npy, E)
    assert E.shape[0] == 176
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    # الوحدات: كل زوج توائم وحدة واحدة
    twin_of = {}
    for a, b in TWINS: twin_of[a] = twin_of[b] = (a, b)
    units, seen = [], set()
    for i, r in d.iterrows():
        key = (r.doc_key, twin_of[r.article_no]) if r.doc_key == "reg" and r.article_no in twin_of else (r.doc_key, (r.article_no,))
        if key in seen: continue
        seen.add(key)
        idx = [d.index[(d.doc_key == key[0]) & (d.article_no == n)][0] for n in key[1]]
        topics = sorted(set(d.loc[idx, "primary_topic"]))
        units.append({"doc_key": key[0], "articles": "/".join(map(str, key[1])), "idx": idx,
                      "primary_topic": topics[0] if len(topics) == 1 else None, "topics_if_twins_differ": " | ".join(topics) if len(topics) > 1 else ""})
    assert len(units) == 150, len(units)
    X = np.vstack([E[u["idx"]].mean(axis=0) for u in units]); X = X / np.linalg.norm(X, axis=1, keepdims=True)
    # عدد المجموعات بـ silhouette
    sil = []
    for k in K_RANGE:
        lab = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit_predict(X)
        sil.append((k, silhouette_score(X, lab, metric="cosine")))
    sil = pd.DataFrame(sil, columns=["k", "silhouette"]); sil.to_csv(out_prefix + "cluster_silhouette.csv", index=False, encoding="utf-8-sig")
    k = int(sil.loc[sil.silhouette.idxmax(), "k"]); best = float(sil.silhouette.max())
    print(f"k المختار: {k} (silhouette {best:.4f}) | أعلى 5:", ", ".join(f"{int(r.k)}:{r.silhouette:.4f}" for r in sil.nlargest(5, "silhouette").itertuples()))
    if k in (min(K_RANGE), max(K_RANGE)): print(f"تنبيه: k المختار ({k}) على طرف المدى المجرَّب {min(K_RANGE)}–{max(K_RANGE)}؛ قد يكون الأعلى خارجه")
    lab = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit_predict(X) + 1
    U = pd.DataFrame([{k_: v for k_, v in u.items() if k_ != "idx"} for u in units]); U["cluster"] = lab
    U.insert(0, "الوثيقة", U.doc_key.map(DOC))
    U.to_csv(out_prefix + "clusters_bge_m3.csv", index=False, encoding="utf-8-sig")
    # المقارنة بالمواضيع
    C = U[U.primary_topic.notna()]
    excl = U[U.primary_topic.isna()]
    for r in excl.itertuples(): print(f"تنبيه: التوأمان اللائحة م{r.articles} بموضوعين مختلفين ({r.topics_if_twins_differ}) — مستبعدان من المقارنة فقط")
    ari = adjusted_rand_score(C.primary_topic, C.cluster)
    ct = pd.crosstab(C.primary_topic, C.cluster); ct.columns = [f"مجموعة {c}" for c in ct.columns]
    print(f"ARI مع primary_topic: {ari:.4f} | وحدات المقارنة: {len(C)} من 150 | عدد المواضيع: {C.primary_topic.nunique()}")
    summ = pd.DataFrame([["الوحدات (التوائم وحدة واحدة)", 150], ["k المختار (أعلى silhouette)", k], ["silhouette عند k", round(best, 4)],
                         ["ARI مع primary_topic", round(ari, 4)], ["وحدات المقارنة", len(C)], ["مستبعدة من المقارنة", ", ".join("اللائحة م" + a for a in excl.articles) or "-"],
                         ["عدد المواضيع في المقارنة", C.primary_topic.nunique()]], columns=["البند", "القيمة"])
    clist = pd.DataFrame({c_: pd.Series({"عدد الوحدات": len(g), "حوكمة": (g.doc_key == "gov").sum(), "لائحة": (g.doc_key == "reg").sum(),
             "أكثر موضوع": g.primary_topic.value_counts().index[0] if g.primary_topic.notna().any() else "", "نسبته": round(g.primary_topic.value_counts(normalize=True).iloc[0], 3) if g.primary_topic.notna().any() else None,
             "المواد": "، ".join(f"{DOC[x.doc_key]} م{x.articles}" for x in g.itertuples())}) for c_, g in U.groupby("cluster")}).T
    clist.index.name = "المجموعة"
    with pd.ExcelWriter(out_prefix + "clusters_vs_topics.xlsx") as w:
        summ.to_excel(w, sheet_name="الملخص", index=False)
        clist.to_excel(w, sheet_name="المجموعات")
        ct.to_excel(w, sheet_name="تقاطع المواضيع والمجموعات")
        sil.to_excel(w, sheet_name="silhouette", index=False)
    import openpyxl
    wb = openpyxl.load_workbook(out_prefix + "clusters_vs_topics.xlsx")
    for ws in wb.worksheets: ws.sheet_view.rightToLeft = True
    wb.save(out_prefix + "clusters_vs_topics.xlsx")
    print("حُفظ: bge_m3_embeddings.npy، cluster_silhouette.csv، clusters_bge_m3.csv، clusters_vs_topics.xlsx")
    return U, sil, ari

U, sil, ari = main(CSV, from_npy=True)                 # يستعمل تمثيلات bge-m3 المحفوظة في الخطوة 2

## تنزيل المخرجات

In [ ]:
outs = ['tfidf_matches.csv', 'bge_m3_matches.csv', 'bge_m3_embeddings.npy', 'cluster_silhouette.csv', 'clusters_bge_m3.csv', 'clusters_vs_topics.xlsx']
with zipfile.ZipFile('/content/model_outputs.zip', 'w') as z:
    for f in outs: z.write(f)
files.download('/content/model_outputs.zip')

## 4) العرض: اكتبي مادة، ويقترح النموذج أقرب مواد اللائحة التنفيذية
المقترحات للاسترشاد، والحكم النهائي بقراءة نص المادة. دقة النموذج مقاسة على مواد قواعد الحوكمة فقط؛ على نصوص أخرى لم تُقَس.
مثال نتيجته معروفة: «يتولى المركز تفسير هذه القواعد, ويكون تفسيره ملزما.» (الحوكمة م46) ← اللائحة م128 بدرجة نحو 0.856.

In [ ]:
# تنظيف النص المكتوب بالطريقة نفسها التي نُظّف بها text_raw: حذف التشكيل وعلامات الاتجاه الخفية، وأرقام لاتينية
DIAC=re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭـ]")
BIDI=re.compile(r"[‎‏‪-‮⁦-⁩﻿؜]")
DIG=str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹","01234567890123456789")
clean = lambda t: BIDI.sub('', DIAC.sub('', t)).translate(DIG).strip()
def top3(text):
    s = R @ enc([clean(text)])[0]
    seen, out = set(), []
    for j in np.argsort(-s):
        n = int(reg.article_no[j]); k = rep.get(n, str(n))
        if k in seen: continue
        seen.add(k); out.append((k, float(s[j]), j))
        if len(out) == 3: break
    return out

import ipywidgets as w
from IPython.display import display, HTML
box = w.Textarea(placeholder='الصقي أو اكتبي نص المادة هنا…', layout=w.Layout(width='100%', height='160px'))
btn = w.Button(description='طابقي', button_style='primary'); out = w.Output()
def show(_):
    out.clear_output()
    t = box.value.strip()
    with out:
        if len(t) < 20: print('اكتبي نص مادة كاملًا (20 حرفًا على الأقل).'); return
        rows = ''
        for r, (k, sc, j) in enumerate(top3(t), 1):
            ok = sc >= 0.75
            txt = html.escape(reg.text_raw[j][:350]).replace('\n', '<br>') + ('…' if len(reg.text_raw[j]) > 350 else '')
            rows += (f'<tr><td>{r}</td><td>المادة {k}</td><td>{sc:.3f}</td>'
                     f'<td style="color:{"#1a7f37" if ok else "#9a6700"}">{"نعم" if ok else "لا"}</td><td style="text-align:right">{txt}</td></tr>')
        display(HTML('<div dir="rtl" style="font-family:sans-serif"><table border="1" cellpadding="6" style="border-collapse:collapse">'
                     '<tr><th>الترتيب</th><th>مادة اللائحة</th><th>درجة التشابه</th><th>فوق الحد 0.75</th><th>بداية نص المادة</th></tr>'
                     + rows + '</table><p style="color:#555">مقترحات النموذج للاسترشاد؛ فوق 0.75 كانت 91% من المقترحات صحيحة في المراجعة البشرية. الحكم النهائي بقراءة النص.</p></div>'))
btn.on_click(show)
display(box, btn, out)